In [3]:
# Cell 00: Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

Running as a Colab notebook
Dependencies installed


In [ ]:
# Cell 0: Imports
import sys
import torch
from pathlib import Path

# Detect the RUNTIME, not a specific synced subfolder: a Colab sync may bring
# src/ without data/, so probing for /content/data is unreliable. Use the same
# signal Cell 0 uses (import google.colab).
#   Colab: repo is under /content
#   Local: notebook is in notebooks/, project root is one level up
try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content')
except ImportError:
    PROJECT_ROOT = Path.cwd().parent

# src/ must be importable under BOTH kernels (local + Colab).
sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

print(f"Project root: {PROJECT_ROOT}")

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cuda


In [13]:
# ← Per-scale run: edit this one string, then Restart & Run All.
model_name = "EleutherAI/pythia-12b"

In [ ]:
model = HookedTransformer.from_pretrained(model_name, device=device)
model.eval()

print(f"n_layers: {model.cfg.n_layers}")
print(f"d_model: {model.cfg.d_model}")
print(f"d_mlp: {model.cfg.d_mlp}")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.11G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.93G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.81G [00:00<?, ?B/s]

In [5]:
prompt = "A long navigation menu without a skip link is not accessible because"
#prompt = "A website without screen reader support is not accessible because"
#prompt = "An image without alt text is not accessible because"

tokens = model.to_tokens(prompt)
print(f"prompt tokens: {tokens.shape}")
print(f"token strings: {model.to_str_tokens(prompt)}")

output = model.generate(prompt, max_new_tokens=50, temperature=0, verbose=False)
print("---")
print(output)

prompt tokens: torch.Size([1, 13])
token strings: ['<|endoftext|>', 'A', ' long', ' navigation', ' menu', ' without', ' a', ' skip', ' link', ' is', ' not', ' accessible', ' because']
---
A long navigation menu without a skip link is not accessible because it is not a link.

A long navigation menu without a skip link is not accessible because it is not a link.

A long navigation menu without a skip link is not accessible because it is not a link.

A long


In [6]:
# Generate with tokens visible one at a time
input_ids = model.to_tokens(prompt)
generated_tokens = []

with torch.no_grad():
    current_ids = input_ids.clone()
    for step in range(10):
        logits = model(current_ids)
        next_token_logits = logits[0, -1, :]
        next_token = torch.argmax(next_token_logits).item()
        next_token_str = model.to_string(torch.tensor([next_token]))
        generated_tokens.append((step, next_token, next_token_str, next_token_logits[next_token].item()))
        current_ids = torch.cat([current_ids, torch.tensor([[next_token]], device=device)], dim=1)

print(f"{'step':<5} {'token_id':<10} {'string':<20} {'logit':<10}")
print("-" * 50)
for step, tok_id, tok_str, logit in generated_tokens:
    print(f"{step:<5} {tok_id:<10} {repr(tok_str):<20} {logit:<10.3f}")

step  token_id   string               logit     
--------------------------------------------------
0     352        ' it'                16.188    
1     310        ' is'                17.084    
2     417        ' not'               16.304    
3     247        ' a'                 14.184    
4     3048       ' link'              14.980    
5     15         '.'                  17.950    
6     187        '\n'                 16.169    
7     187        '\n'                 17.707    
8     34         'A'                  14.423    
9     1048       ' long'              15.215    


In [7]:
# What's competing at the decision point right after "because"?
input_ids = model.to_tokens(prompt)

with torch.no_grad():
    logits = model(input_ids)
    step0_logits = logits[0, -1, :]

# Top 15 candidates
top_values, top_indices = torch.topk(step0_logits, 15)

print(f"{'rank':<6} {'token_id':<10} {'string':<20} {'logit':<10}")
print("-" * 50)
for rank, (val, idx) in enumerate(zip(top_values, top_indices)):
    tok_str = model.to_string(torch.tensor([idx.item()]))
    print(f"{rank:<6} {idx.item():<10} {repr(tok_str):<20} {val.item():<10.3f}")

rank   token_id   string               logit     
--------------------------------------------------
0      352        ' it'                16.188    
1      253        ' the'               16.014    
2      273        ' of'                14.438    
3      627        ' there'             14.226    
4      247        ' a'                 14.092    
5      368        ' you'               13.755    
6      187        '\n'                 13.689    
7      4212       ' users'             13.487    
8      697        ' its'               13.345    
9      309        ' I'                 12.578    
10     642        ' no'                12.464    
11     3601       ' screen'            12.445    
12     27         ':'                  12.323    
13     359        ' we'                12.273    
14     512        ' all'               12.271    


In [8]:
# Cell 8: Mechanical export -> results/logits/
# Supersedes the print -> eyeball -> hand-paste-into-markdown pipeline that
# produced docs/tangent.md. Runs the three VERIFIED prompts only; the five
# DRAFT prompts in logit_export are out of scope (review-before-use).
from src.logit_export import export_because_tables, VERIFIED_PROMPTS

# Filenames key on the short model name (drop the "EleutherAI/" path segment).
short_name = model_name.split("/")[-1]

export_because_tables(
    model,
    short_name,
    VERIFIED_PROMPTS,
    PROJECT_ROOT / "results" / "logits",
)

exported: pythia-6.9b_skip_link_because_ranks.csv, pythia-6.9b_skip_link_because_steps.csv
exported: pythia-6.9b_screen_reader_because_ranks.csv, pythia-6.9b_screen_reader_because_steps.csv
exported: pythia-6.9b_alt_text_because_ranks.csv, pythia-6.9b_alt_text_because_steps.csv
exported: pythia-6.9b_because_generations.csv
